<a href="https://colab.research.google.com/github/khushijindal06/damwork/blob/codex/add-damwork-notebook/damwork.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Duan UAV RGB piping: clean vs degraded YOLO experiment

This notebook implements the complete requested workflow:

1. Verify Duan et al.'s open Zenodo dataset.
2. Convert the rendered bounding rectangles to YOLO labels.
3. Remove only the rendered annotation overlay to create a **derived clean** RGB set.
4. Create leakage-free train/validation/test splits by site/date capture group.
5. Generate haze, fog, low-light, and mixed variants **after splitting**.
6. Save synthetic images under a separate `synthetic` folder.
7. Train clean-only and degradation-robust YOLO models.
8. Evaluate clean, degraded, and enhanced test sets.
9. Measure how much mAP50-95 each enhancement method recovers.

Official source: [Zenodo record 10896178](https://zenodo.org/records/10896178), DOI
`10.5281/zenodo.10896178`, CC BY 4.0.


## Important source-data fact

The official archive contains visible and infrared JPGs with `Piping` text and
bounding rectangles already drawn into the pixels; it does not include XML,
JSON, or YOLO sidecar annotations.

The preparation script therefore extracts the displayed rectangle, writes a
YOLO label, and inpaints only the text and rectangle border. Original Zenodo
files are never modified. Review the generated audit samples before training.


## 0. Set up the repository and dependencies

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo = Path("/content/damwork")
    if not repo.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--branch",
                "codex/add-damwork-notebook",
                "--single-branch",
                "https://github.com/khushijindal06/damwork.git",
                str(repo),
            ],
            check=True,
        )
    os.chdir(repo)
else:
    candidates = [Path.cwd(), Path.cwd() / "work" / "damwork"]
    repo = next(
        (path for path in candidates if (path / "DamVis" / "scripts").exists()),
        None,
    )
    if repo is None:
        raise FileNotFoundError("Run this notebook from the damwork repository.")
    os.chdir(repo)

REPO_ROOT = Path.cwd().resolve()
print("Repository:", REPO_ROOT)


In [ ]:
# Run once in a fresh kernel.
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(REPO_ROOT / "DamVis" / "requirements-yolo.txt"),
    ],
    check=True,
)


## 1. Define source and output folders

In [ ]:
# Local Windows defaults requested for this project.
WINDOWS_SOURCE = Path(
    r"E:/Downloads/10896178/UAV_piping_image/UAV_piping_label_data/Visible"
)
WINDOWS_EXPERIMENT = Path(r"E:/Downloads/10896178/Duan_RGB_Experiment")

# Override either path before running this cell with environment variables:
# DUAN_VISIBLE_SOURCE and DUAN_EXPERIMENT_ROOT.
if IN_COLAB:
    SOURCE_DIR = Path(os.environ.get("DUAN_VISIBLE_SOURCE", "/content/duan_source/Visible"))
    EXPERIMENT_ROOT = Path(
        os.environ.get("DUAN_EXPERIMENT_ROOT", "/content/duan_rgb_experiment")
    )
else:
    SOURCE_DIR = Path(os.environ.get("DUAN_VISIBLE_SOURCE", WINDOWS_SOURCE))
    EXPERIMENT_ROOT = Path(
        os.environ.get("DUAN_EXPERIMENT_ROOT", WINDOWS_EXPERIMENT)
    )

CLEAN_ROOT = EXPERIMENT_ROOT / "clean"
SYNTHETIC_ROOT = EXPERIMENT_ROOT / "synthetic"
ENHANCED_ROOT = EXPERIMENT_ROOT / "enhanced"
METADATA_ROOT = EXPERIMENT_ROOT / "metadata"

print("Visible source :", SOURCE_DIR)
print("Experiment root:", EXPERIMENT_ROOT)
print("Clean output   :", CLEAN_ROOT)
print("Synthetic output (separate folder):", SYNTHETIC_ROOT)
print("Enhanced output:", ENHANCED_ROOT)


### Colab only: download the official Zenodo archive

In [ ]:
if IN_COLAB and not SOURCE_DIR.exists():
    import hashlib
    import urllib.request
    import zipfile

    archive = Path("/content/UAV_piping_image.zip")
    url = (
        "https://zenodo.org/api/records/10896178/files/"
        "UAV_piping_image.zip/content"
    )
    urllib.request.urlretrieve(url, archive)

    md5 = hashlib.md5(archive.read_bytes()).hexdigest()
    expected = "41a65d1914a0649bc96285211bdc83a0"
    if md5 != expected:
        raise RuntimeError(f"Archive checksum mismatch: {md5}")

    extract_root = Path("/content/duan_source")
    with zipfile.ZipFile(archive) as bundle:
        bundle.extractall(extract_root)

    matches = list(extract_root.rglob("UAV_piping_label_data/Visible"))
    if len(matches) != 1:
        raise RuntimeError(f"Expected one visible source folder, found {matches}")
    SOURCE_DIR = matches[0]
    print("Downloaded and verified:", SOURCE_DIR)
else:
    print("Using existing source:", SOURCE_DIR)


## 2. Verify the RGB source before conversion

In [ ]:
from collections import Counter
import re

source_images = sorted(SOURCE_DIR.glob("*.jpg"))
if not source_images:
    raise FileNotFoundError(f"No visible JPG images found in {SOURCE_DIR}")

groups = Counter()
for path in source_images:
    match = re.match(r"^P_([A-Z]{2}\d{6})_", path.stem)
    if not match:
        raise ValueError(f"Unexpected filename: {path.name}")
    groups[match.group(1)] += 1

print("Visible RGB images:", len(source_images))
print("Site/date capture groups:", len(groups))
print(dict(sorted(groups.items())))

if len(source_images) != 625:
    print("WARNING: the official release normally contains 625 visible RGB images.")


## 3. Convert rendered boxes to YOLO and create derived clean images

In [ ]:
PREPARE_SCRIPT = REPO_ROOT / "DamVis" / "scripts" / "07_prepare_duan_yolo.py"

subprocess.run(
    [
        sys.executable,
        str(PREPARE_SCRIPT),
        "--source",
        str(SOURCE_DIR),
        "--output",
        str(EXPERIMENT_ROOT),
        "--reset",
        "--audit-samples",
        "30",
    ],
    check=True,
)


## 4. Audit conversion and leakage-free splits

In [ ]:
import json
import pandas as pd

preparation_report = json.loads(
    (METADATA_ROOT / "preparation_report.json").read_text(encoding="utf-8")
)
display(preparation_report)

assert preparation_report["conversion_failures"] == 0
assert preparation_report["leakage_audit"]["leaking_capture_groups"] == []
assert preparation_report["leakage_audit"]["cross_split_exact_hashes"] == []

manifest = pd.read_csv(METADATA_ROOT / "clean_manifest.csv")
display(
    manifest.groupby(["split", "group"]).size().rename("images").reset_index()
)
display(manifest.groupby("split").size().rename("images"))


In [ ]:
# Review several original-vs-derived-clean audit pairs.
from IPython.display import Image as DisplayImage, display

audit_paths = sorted((METADATA_ROOT / "audit_samples").glob("*.jpg"))
for path in audit_paths[:8]:
    print(path.name, "(left: source with detected box, right: derived clean)")
    display(DisplayImage(filename=str(path), width=900))


**Stop here if any red audit rectangle is wrong.** Training on incorrect
boxes or leaving the rendered label visible can create a shortcut and invalidate
the experiment.


## 5. Generate 12 synthetic variants in the separate folder

In [ ]:
SYNTHETIC_SCRIPT = (
    REPO_ROOT / "DamVis" / "scripts" / "08_generate_duan_synthetic.py"
)

subprocess.run(
    [
        sys.executable,
        str(SYNTHETIC_SCRIPT),
        "--experiment",
        str(EXPERIMENT_ROOT),
        "--reset",
    ],
    check=True,
)


In [ ]:
synthetic_report = json.loads(
    (METADATA_ROOT / "synthetic_report.json").read_text(encoding="utf-8")
)
display(synthetic_report)

synthetic_manifest = pd.read_csv(METADATA_ROOT / "synthetic_manifest.csv")
display(
    synthetic_manifest.groupby(["split", "family", "variant"])
    .size()
    .rename("images")
    .reset_index()
)

assert Path(synthetic_report["synthetic_root"]) == SYNTHETIC_ROOT.resolve()
assert synthetic_report["labels_preserved_byte_for_byte"] is True
assert synthetic_report["geometry_preserved"] is True


## 6. Generate enhanced degraded test sets

In [ ]:
ENHANCEMENT_SCRIPT = (
    REPO_ROOT / "DamVis" / "scripts" / "09_enhance_duan_test.py"
)

subprocess.run(
    [
        sys.executable,
        str(ENHANCEMENT_SCRIPT),
        "--experiment",
        str(EXPERIMENT_ROOT),
        "--reset",
        "--methods",
        "clahe",
        "gamma",
        "retinex",
        "dcp",
    ],
    check=True,
)


In [ ]:
enhancement_report = json.loads(
    (METADATA_ROOT / "enhancement_report.json").read_text(encoding="utf-8")
)
display(enhancement_report)


## 7. Train clean-only and robust YOLO models

The robust model trains on clean plus synthetic **train** images and
validates on clean plus synthetic **validation** images. Synthetic test images
and all enhanced images remain evaluation-only.

For a quick pipeline check, set `EPOCHS = 1`. Use at least 50 epochs for the
reported experiment and keep the same seed/model settings for both runs.


In [ ]:
TRAIN_SCRIPT = (
    REPO_ROOT / "DamVis" / "scripts" / "10_train_evaluate_duan_yolo.py"
)

EPOCHS = 50
MODEL = "yolo11n.pt"
DEVICE = 0 if IN_COLAB else None

train_command = [
    sys.executable,
    str(TRAIN_SCRIPT),
    "--experiment",
    str(EXPERIMENT_ROOT),
    "--model",
    MODEL,
    "--epochs",
    str(EPOCHS),
    "--batch",
    "16",
    "--imgsz",
    "640",
]
if DEVICE is not None:
    train_command.extend(["--device", str(DEVICE)])

subprocess.run(train_command, check=True)


## 8. Compare clean, degraded, and enhanced mAP

In [ ]:
metrics = pd.read_csv(METADATA_ROOT / "yolo_metrics.csv")
recovery = pd.read_csv(METADATA_ROOT / "yolo_recovery.csv")

display(metrics.sort_values(["model", "condition"]))
display(
    recovery.sort_values(
        ["model", "variant", "recovery_fraction"],
        ascending=[True, True, False],
    )
)


In [ ]:
import matplotlib.pyplot as plt

degraded_only = metrics[
    ~metrics["condition"].str.contains("/")
    & (metrics["condition"] != "clean")
]

fig, axes = plt.subplots(1, 2, figsize=(17, 6))
for model_name, frame in degraded_only.groupby("model"):
    axes[0].plot(
        frame["condition"],
        frame["map50_95"],
        marker="o",
        label=model_name,
    )
axes[0].set_title("YOLO mAP50-95 on degraded test conditions")
axes[0].set_ylabel("mAP50-95")
axes[0].tick_params(axis="x", rotation=75)
axes[0].legend()
axes[0].grid(alpha=0.25)

best_recovery = (
    recovery.groupby(["model", "variant"], as_index=False)
    .apply(lambda frame: frame.loc[frame["recovery_fraction"].idxmax()])
    .reset_index(drop=True)
)
for model_name, frame in best_recovery.groupby("model"):
    axes[1].bar(
        frame["variant"] + " / " + model_name,
        frame["recovery_fraction"],
        label=model_name,
    )
axes[1].axhline(0, color="black", linewidth=1)
axes[1].axhline(1, color="green", linewidth=1, linestyle="--")
axes[1].set_title("Best enhancement recovery fraction per condition")
axes[1].set_ylabel("0 = no recovery; 1 = clean mAP restored")
axes[1].tick_params(axis="x", rotation=80)
axes[1].grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


## Interpretation checklist

- Compare the clean-only model's clean mAP with each degraded condition.
- Compare clean-only vs robust model under the same degraded condition.
- Enhancement is useful only when `absolute_gain > 0`.
- `recovery_fraction = 1` means the enhanced condition restored the clean-test
  mAP gap; values above 1 exceed the clean baseline.
- Report negative recovery values too; enhancement can damage detector features.
- Do not tune enhancement parameters on the test set. Use validation data for
  any future parameter selection.
